In [1]:
# Generate sets of differential equations for single qubits and multiple qubits in a cavity up to n'th operator order using the hierarchical equations of motion approach
# Finally use cumulants to make the problem solvable
include("operator_terms.jl")
include("combinatorics.jl")
include("cumulants.jl")
include("diff_Eq.jl")
include("preprocessing.jl")
include("indexing.jl")
include("initial_states.jl")
include("pulses.jl")
include("diff_Eq_solver.jl")
println("Included all files")
#using ForwardDiff
#using Optimization, Ipopt, OptimizationOptimJL 
using Random
using Plots 
rng = MersenneTwister(1234)

Included all files


MersenneTwister(1234)

In [2]:
############################################################################
#### Single Qubit in a cavity ##############################################
############################################################################

In [3]:
n = 2
all_eqs = single_spin_gen_DE_up_to_order(n)
conjugate_to_basis!(all_eqs)           # Add this line to replace terms outside of the basis with conjugate versions in the basis
for eqs in all_eqs
    for eq in eqs
        display(eq)
    end
end

d/dt⟨a⟩ =  - κ/2κ⟨a⟩ - g/2(i⟨x⟩ + ⟨y⟩) + β√̅κ

d/dt⟨x⟩ = ig(⟨az⟩ - ⟨az⟩*) - 2γγ⟨x⟩ - ΔΔ⟨y⟩ - Γ/2Γ⟨x⟩

d/dt⟨y⟩ =  - g(⟨az⟩ + ⟨az⟩*) - 2γγ⟨y⟩ + ΔΔ⟨x⟩ - Γ/2Γ⟨y⟩

d/dt⟨z⟩ = g(⟨ay⟩ - i⟨ax⟩ + ⟨ay⟩* + i⟨ax⟩*) - Γ(⟨z⟩ + 1)

d/dt⟨a²⟩ =  - κκ⟨a²⟩ - g(i⟨ax⟩ + ⟨ay⟩) + 2β√̅κβ√̅κ⟨a⟩

d/dt⟨a†a⟩ =  - κκ⟨a†a⟩ + g/2( - i⟨ax⟩* - ⟨ay⟩* + i⟨ax⟩ - ⟨ay⟩) + β√̅κβ√̅κ⟨a⟩* + β*√̅κβ*√̅κ⟨a⟩

d/dt⟨ax⟩ =  - κ/2κ⟨ax⟩ + ig/2(2⟨a²z⟩ - 1 - 2⟨a†az⟩ - ⟨z⟩) + β√̅κβ√̅κ⟨x⟩ - 2γγ⟨ax⟩ - ΔΔ⟨ay⟩ - Γ/2Γ⟨ax⟩

d/dt⟨ay⟩ =  - κ/2κ⟨ay⟩ - g/2(2⟨a²z⟩ + 2⟨a†az⟩ + ⟨z⟩ + 1) + β√̅κβ√̅κ⟨y⟩ - 2γγ⟨ay⟩ + ΔΔ⟨ax⟩ - Γ/2Γ⟨ay⟩

d/dt⟨az⟩ =  - κ/2κ⟨az⟩ + g/2(2⟨a²y⟩ - 2i⟨a²x⟩ + 2⟨a†ay⟩ + ⟨y⟩ + 2i⟨a†ax⟩ + i⟨x⟩) + β√̅κβ√̅κ⟨z⟩ - Γ(⟨az⟩ + ⟨a⟩)

In [4]:
#### Calculate Cumulant terms from cumulant_indexed_vector and an operator expectation value vector
# Test
p::Float64 = 0.1
alpha::Float64 = 0.1
Theta::Float64 = 0.1
all_eqs = single_spin_gen_DE_up_to_order(3)
conjugate_to_basis!(all_eqs)
all_eqs_indexed, op_index_dicts, vars_dict, cumulant_terms_dict = parse_DE_to_indexes(all_eqs)
cumulant_vector = cumulant_terms_from_dict(cumulant_terms_dict, op_index_dicts)
lower_cumulants = lower_order_full_cumulant_terms(op_index_dicts, 2)
op_index_vec, vars_vec, cumulant_terms_vec = preprocess_index_inversion(op_index_dicts, vars_dict, cumulant_terms_dict)
initial_conditions = equations2initial_conditions(op_index_vec, p, alpha, Theta)
display(cumulant_vector)
display(lower_cumulants)
display(get_cumulant_values_provided(cumulant_vector, initial_conditions))

6-element Vector{Cumulant_indexed}:
 Cumulant_indexed([[1, 1, 1, 4], [5, 1, 4], [9, 1, 1], [5, 9], [9, 5], [10, 4], [16, 1]], [6, -6, -6, 2, 1, 1, 3], "---z")
 Cumulant_indexed([[-1, 1, 1, 4], [6, 1, 4], [-9, 1, 1], [5, -1, 4], [9, -1, 1], [6, 9], [-9, 5], [11, 4], [17, 1], [16, -1]], [6, -4, -2, -2, -4, 2, 1, 1, 2, 1], "+--z")
 Cumulant_indexed([[1, 1, 1, 3], [5, 1, 3], [8, 1, 1], [5, 8], [8, 5], [10, 3], [14, 1]], [6, -6, -6, 2, 1, 1, 3], "---y")
 Cumulant_indexed([[1, 1, 1, 2], [5, 1, 2], [7, 1, 1], [5, 7], [7, 5], [10, 2], [12, 1]], [6, -6, -6, 2, 1, 1, 3], "---x")
 Cumulant_indexed([[-1, 1, 1, 3], [6, 1, 3], [-8, 1, 1], [5, -1, 3], [8, -1, 1], [6, 8], [-8, 5], [11, 3], [15, 1], [14, -1]], [6, -4, -2, -2, -4, 2, 1, 1, 2, 1], "+--y")
 Cumulant_indexed([[-1, 1, 1, 2], [6, 1, 2], [-7, 1, 1], [5, -1, 2], [7, -1, 1], [6, 7], [-7, 5], [11, 2], [13, 1], [12, -1]], [6, -4, -2, -2, -4, 2, 1, 1, 2, 1], "+--x")

3-element Vector{Vector{Vector{Full_Cumulant_indexed}}}:
 [[Full_Cumulant_indexed([[1]], [1], "-")], [Full_Cumulant_indexed([[4]], [1], "z"), Full_Cumulant_indexed([[3]], [1], "y")]]
 [[Full_Cumulant_indexed([[-1, 1], [6]], [-1, 1], "+-"), Full_Cumulant_indexed([[1, 1], [5]], [-1, 1], "--")], [Full_Cumulant_indexed([[1, 4], [9]], [-1, 1], "-z"), Full_Cumulant_indexed([[1, 3], [8]], [-1, 1], "-y")]]
 [[Full_Cumulant_indexed([[-1, 1, 1], [6, 1], [5, -1], [11]], [2, -2, -1, 1], "+--"), Full_Cumulant_indexed([[1, 1, 1], [5, 1], [10]], [2, -3, 1], "---")], [Full_Cumulant_indexed([[-1, 1, 4], [6, 4], [-9, 1], [9, -1], [17]], [2, -1, -1, -1, 1], "+-z"), Full_Cumulant_indexed([[-1, 1, 3], [6, 3], [-8, 1], [8, -1], [15]], [2, -1, -1, -1, 1], "+-y")]]

6-element Vector{ComplexF64}:
 0.0007642691913004843 - 0.00023641616532907176im
 0.0007960033322224217 - 7.986673331746262e-5im
                   0.0 + 0.0im
                   0.0 + 0.0im
                   0.0 + 0.0im
                   0.0 + 0.0im

In [5]:
op_index_vec

17-element Vector{String}:
 "-"
 "x"
 "y"
 "z"
 "--"
 "+-"
 "-x"
 "-y"
 "-z"
 "---"
 "+--"
 "--x"
 "+-x"
 "--y"
 "+-y"
 "--z"
 "+-z"

In [11]:
how_many::Int = 10
max_A::Float64 =  0.1
T::Float64 = 10.0
n = 10
pulse_param::Pulse_Param_Struct = Random_Pulse_Param_Wurst(how_many, max_A, T, n, rng=rng)
val_dict::Dict = Dict("\\beta" => wurst_pulse_generator(pulse_param), "\\Delta" => 0.1, "g" => 0.2, "\\gamma" => 0.1, "\\Gamma" => 0.1, "\\kappa" => 0.1)

Dict{String, Any} with 6 entries:
  "g"       => 0.2
  "\\Gamma" => 0.1
  "\\gamma" => 0.1
  "\\beta"  => fun
  "\\Delta" => 0.1
  "\\kappa" => 0.1

In [20]:
param_generator, constant_value_indexes = value_generator(pulse_param, val_dict, vars_vec)
all_eqs_indexed_reduced = remove_zero_terms(all_eqs_indexed, param_generator, constant_value_indexes; how_many_eps=5)
diff_problem = DifferentialProblem(param_generator, all_eqs_indexed_reduced, cumulant_vector, Full_Cumulant_indexed[])
#d_initial_conditions = zeros(ComplexF64, length(op_index_vec))
solution = evolve_system(initial_conditions, diff_problem, T, dt=0.01, autodiff=true, reltol=1e-10)
println("Done")

Done


In [27]:
# Generate sets of differential equations for single qubits and multiple qubits in a cavity up to n'th operator order using the hierarchical equations of motion approach
# Finally use cumulants to make the problem solvable
include("operator_terms.jl")
include("combinatorics.jl")
include("cumulants.jl")
include("diff_Eq.jl")
include("preprocessing.jl")
include("indexing.jl")
include("initial_states.jl")
include("pulses.jl")
#include("diff_Eq_solver.jl")
println("Included all files")
#using ForwardDiff
#using Optimization, Ipopt, OptimizationOptimJL 
using Random
using Plots 
rng = MersenneTwister(1234)

Included all files


MersenneTwister(1234)

In [28]:
############################################################################
#### Multi Qubit in a cavity ###############################################
############################################################################

In [29]:
n = 2
all_eqs = multi_spin_gen_DE_up_to_order(n)
conjugate_to_basis!(all_eqs)           # Add this line to replace terms outside of the basis with conjugate versions in the basis
our_eq = nothing
for eqs in all_eqs
    for eq in eqs
        #if term2str_identifier(eq.exp_op)[1] == "-xx"
        display(eq)
        #our_eq = eq
        #end
    end
end

d/dt⟨a⟩ = ∑ᵢ - g_i/2(i⟨xᵢ⟩ + ⟨yᵢ⟩) +  - κ/2κ⟨a⟩ + β√̅κ

d/dt⟨xᵢ⟩ = ig_i(⟨azᵢ⟩ - ⟨azᵢ⟩*) - Δ_iΔ_i⟨yᵢ⟩ - 2γγ⟨xᵢ⟩ - Γ/2Γ⟨xᵢ⟩

d/dt⟨yᵢ⟩ =  - g_i(⟨azᵢ⟩ + ⟨azᵢ⟩*) + Δ_iΔ_i⟨xᵢ⟩ - 2γγ⟨yᵢ⟩ - Γ/2Γ⟨yᵢ⟩

d/dt⟨zᵢ⟩ = g_i(⟨ayᵢ⟩ - i⟨axᵢ⟩ + ⟨ayᵢ⟩* + i⟨axᵢ⟩*) - Γ(⟨zᵢ⟩ + 1)

d/dt⟨a²⟩ = ∑ᵢ - g_i(i⟨axᵢ⟩ + ⟨ayᵢ⟩) +  - κκ⟨a²⟩ + 2β√̅κβ√̅κ⟨a⟩

...

d/dt⟨yᵢyⱼ⟩ = Δ_jΔ_j⟨yᵢxⱼ⟩ - g_i(⟨azᵢyⱼ⟩ + ⟨azᵢyⱼ⟩*) + Δ_iΔ_i⟨xᵢyⱼ⟩ - 4γγ⟨yᵢyⱼ⟩ - g_j(⟨ayᵢzⱼ⟩ + ⟨ayᵢzⱼ⟩*) - ΓΓ⟨yᵢyⱼ⟩

d/dt⟨yᵢzⱼ⟩ =  - g_i(⟨azᵢzⱼ⟩ + ⟨azᵢzⱼ⟩*) + Δ_iΔ_i⟨xᵢzⱼ⟩ - 2γγ⟨yᵢzⱼ⟩ + g_j(⟨ayᵢyⱼ⟩ - i⟨ayᵢxⱼ⟩ + ⟨ayᵢyⱼ⟩* + i⟨ayᵢxⱼ⟩*) + Γ( - 3/2⟨yᵢzⱼ⟩ - ⟨yᵢ⟩)

d/dt⟨zᵢxⱼ⟩ =  - Δ_jΔ_j⟨zᵢyⱼ⟩ + g_i(⟨ayᵢxⱼ⟩ - i⟨axᵢxⱼ⟩ + ⟨ayᵢxⱼ⟩* + i⟨axᵢxⱼ⟩*) - 2γγ⟨zᵢxⱼ⟩ + ig_j(⟨azᵢzⱼ⟩ - ⟨azᵢzⱼ⟩*) + Γ( - 3/2⟨zᵢxⱼ⟩ - ⟨xⱼ⟩)

d/dt⟨zᵢyⱼ⟩ = Δ_jΔ_j⟨zᵢxⱼ⟩ + g_i(⟨ayᵢyⱼ⟩ - i⟨axᵢyⱼ⟩ + ⟨ayᵢyⱼ⟩* + i⟨axᵢyⱼ⟩*) - 2γγ⟨zᵢyⱼ⟩ - g_j(⟨azᵢzⱼ⟩ + ⟨azᵢzⱼ⟩*) + Γ( - 3/2⟨zᵢyⱼ⟩ - ⟨yⱼ⟩)

d/dt⟨zᵢzⱼ⟩ = g_i(⟨ayᵢzⱼ⟩ - i⟨axᵢzⱼ⟩ + ⟨ayᵢzⱼ⟩* + i⟨axᵢzⱼ⟩*) + g_j(⟨azᵢyⱼ⟩ - i⟨azᵢxⱼ⟩ + ⟨azᵢyⱼ⟩* + i⟨azᵢxⱼ⟩*) - Γ(2⟨zᵢzⱼ⟩ + ⟨zⱼ⟩ + ⟨zᵢ⟩)

In [30]:
n = 3
all_eqs = multi_spin_gen_DE_up_to_order(n)
conjugate_to_basis!(all_eqs)
all_eqs_indexed, op_index_dicts, vars_dict, cumulant_terms_dict = parse_DE_to_indexes(all_eqs)
op_index_vec, vars_vec, cumulant_terms_vec = preprocess_index_inversion(op_index_dicts, vars_dict, cumulant_terms_dict)

(["-", "x", "y", "z", "--", "+-", "-x", "-y", "-z", "xx"  …  "yzz", "zxx", "zxy", "zxz", "zyx", "zyy", "zyz", "zzx", "zzy", "zzz"], [["g"], ["\\kappa"], ["\\beta", "\\sqrt{\\kappa}"], ["\\Delta"], ["\\gamma"], ["\\Gamma"], ["\\beta^{*}", "\\sqrt{\\kappa}"]], ["---z", "+--z", "---y", "---x", "+--y", "+--x", "--zx", "+-zx", "--xz", "+-xz"  …  "-zzz", "-yzx", "-yxz", "-yzy", "-yzz", "-yxy", "-yxx", "-yyz", "-yyy", "-yyx"])

In [32]:
# given an operator , give all indexes to operators that change the position of the first spin operator in the operator
function all_i_neq_j_str_combinations(term_str::String)::Tuple{Vector{String}, Int}
    # find the position of the first spin operator
    first_spin_pos::Int = 0
    for i in 1:length(term_str)
        if term_str[i] == 'x' || term_str[i] == 'y' || term_str[i] == 'z'
            first_spin_pos = i
            break
        end
    end
    if first_spin_pos == 0
        error("No spin operator found in term")   # check other code, why did you even evaluate this?!?
    end
    # number of spin operators
    N_s = count(i -> (i=='x'), term_str) + count(i -> (i=='y'), term_str) + count(i -> (i=='z'), term_str)
    #construct all variations, i.e. if +-xyz -> +-xyz, +-yxz, +-yzx ( all positions of the first spin operator)
    first_spin_combinations::Vector{String} = Vector{String}(undef, N_s)
    first_spin_combinations[1] = term_str
    str_before = term_str[1:first_spin_pos-1]
    str_after = term_str[first_spin_pos+1:end]
    str_op = term_str[first_spin_pos]
    for i in 2:N_s
        first_spin_combinations[i] = str_before*str_after[1:i-1]*str_op*str_after[i:end]
    end
    return first_spin_combinations, N_s
end
function all_i_neq_combinations(term_str::String, op_index_dicts::Vector{Vector{Dict{String,Int}}})::Tuple{Vector{Int}, Int}
    # give first_index_for_op and how_many_indexes_for_io for every combination of position of the first spin operator
    N = length(term_str)
    first_spin_combinations::Vector{String}, N_s = all_i_neq_j_str_combinations(term_str)
    # find the combinations in the dictionary
    first_index_for_op::Vector{Int} = Vector{Int}(undef, length(first_spin_combinations))
    for i in 1:length(first_spin_combinations)
        first_index_for_op[i] = op_index_dicts[N][N_s+1][first_spin_combinations[i]]
    end

    return first_index_for_op, N_s
end
## Test
#term_str = "-xyz"
#display(all_i_neq_j_str_combinations(term_str))
#display(all_i_neq_combinations(term_str, op_index_dicts))

all_i_neq_combinations (generic function with 1 method)

In [33]:
# a function that finds an integer vector inside an (increasingly ordered) vector of integer vectors
function find_vectors_with_indexes(indexes::Vector{Int}, index_combinations::Vector{Vector{Int}}, n_samples::Int=-1)::Vector{Vector{Int}}
    # categorize position of added index
    if !(length(indexes) == length(index_combinations[1]) -1)
        error("Indexes and index_combinations do not match. The length of indexes must be one less than length of index_combinations")
    end
    if n_samples < 0
        n_samples = index_combinations[end][end]
    end
    how_many = n_samples - length(indexes)
    where_indexes::Vector{Int} = Vector{Int}(undef, how_many)
    curr_ind::Int = 1
    i::Int = 1
    while curr_ind <= how_many
        # does it contain indexes?
        if all(x->x in index_combinations[i], indexes)
            where_indexes[curr_ind] = i
            curr_ind += 1
        end
        i += 1
    end
    # separate into positional groups
    positional_indexes::Vector{Vector{Int}} = Vector{Vector{Int}}(undef, length(indexes)+1)
    how_many_each = [indexes[1]-1; indexes[2:end] .- indexes[1:end-1] .- 1; n_samples - indexes[end]]
    cum_summed = [0;cumsum(how_many_each)]
    for i in 1:length(indexes)+1
        positional_indexes[i] = where_indexes[cum_summed[i]+1:cum_summed[i+1]]
    end
    return positional_indexes
end
## Test (thoroughly tested)
#indexes = [2,5,7]
#indexes = find_vectors_with_indexes(indexes, index_combinations[end])
#for (j, ind) in enumerate(indexes)
#    println("Free Index at Position: ",j, " ", ind, "-"^20)
#    for i in ind
#        println(index_combinations[end][i])
#    end
#end

find_vectors_with_indexes (generic function with 2 methods)

In [34]:
# for every spin index combination (spin_index_combinations(n_samples, len)), i would like to get the spin index combinations with one greater length:
# for len=2 if index_combination = [2,4] and n_samples = 6, then i would like to get [[1,2,4]], [[2,3,4]], [[2,4,5], [2,4,6]], all combinations grouped by position of the added spin operator 
function all_i_neq_j_index_combinations(order_of_indexes::Int, n_samples::Int)::Vector{Vector{Int}}
    # for every lower order combination find all the higher order combinations, that contain it
    # start with a dictionary of all the lower order combinations
    # if index_combinations is an integer it is max_order
    index_combinations_N = spin_index_combinations(n_samples, order_of_indexes)
    index_combinations_N_1 = spin_index_combinations(n_samples, order_of_indexes+1)
    all_positional_indexes = Vector{Vector{Vector{Int}}}(undef, length(index_combinations_N)) # by indexes, positions, combinations
    for (i, indexes) in enumerate(index_combinations_N)
        all_positional_indexes[i] = find_vectors_with_indexes(indexes, index_combinations_N_1, n_samples)
    end
    return all_positional_indexes
end
# Add these indexes to the base index of a i_neq_j term 

all_i_neq_j_index_combinations (generic function with 1 method)

In [35]:
# a function that calculates the number of index combinations of spins, takes op_index_dicts as input, to determine, how many spins are at most in operators
# and constructs real operator indexes for the spin_combinations
# use spin_index_combinations(N_s, choose_n) to construct the increasing index combinations
function indexed_DE_with_sampled_combinations(op_index_vec::Vector{String}, n_samples::Int)
    index_combinations::Vector{Vector{Vector{Int}}} = Vector{Vector{Vector{Int}}}()
    # go through the operator index vector element by element
    first_index_for_op::Vector{Int} = zeros(Int, length(op_index_vec))
    how_many_indexes_for_io::Vector{Int} = zeros(Int, length(op_index_vec))
    N_s::Int = 0
    curr_index::Int = 1
    for (i, op_str) in enumerate(op_index_vec)
        # get orders of op_str
        order::Int = length(op_str)
        # get the number of spins in op_str
        N_s = count(i -> (i=='x'), op_str) + count(i -> (i=='y'), op_str) + count(i -> (i=='z'), op_str)
        while N_s+1 > length(index_combinations)
            # add new spin combinations
            push!(index_combinations, spin_index_combinations(n_samples, length(index_combinations)))
        end
        curr_len = length(index_combinations[N_s+1])
        # add that many 
        how_many_indexes_for_io[i] = curr_len
        first_index_for_op[i] = curr_index
        curr_index += curr_len
    end
    # make dict of op_index_vec
    op_index_dict::Dict{String, Tuple{Int, Int}} = Dict{String, Tuple{Int, Int}}()  # Dict{op_str, (first_index, how_many_indexes)}
    for (i, op_str) in enumerate(op_index_vec)
        op_index_dict[op_str] = (first_index_for_op[i], how_many_indexes_for_io[i])
    end
    # make index combination dictionaries 

    return index_combinations, first_index_for_op, how_many_indexes_for_io
end
# Test
n = 4
n_samples = 20
all_eqs = multi_spin_gen_DE_up_to_order(n)
conjugate_to_basis!(all_eqs)
all_eqs_indexed, op_index_dicts, vars_dict, cumulant_terms_dict = parse_DE_to_indexes(all_eqs)
op_index_vec, vars_vec, cumulant_terms_vec = preprocess_index_inversion(op_index_dicts, vars_dict, cumulant_terms_dict)

index_combinations, first_index_for_op, how_many_indexes_for_io = indexed_DE_with_sampled_combinations(op_index_vec, n_samples)
display(index_combinations)

5-element Vector{Vector{Vector{Int}}}:
 [[]]
 [[1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15], [16], [17], [18], [19], [20]]
 [[1, 2], [1, 3], [1, 4], [1, 5], [1, 6], [1, 7], [1, 8], [1, 9], [1, 10], [1, 11]  …  [16, 17], [16, 18], [16, 19], [16, 20], [17, 18], [17, 19], [17, 20], [18, 19], [18, 20], [19, 20]]
 [[1, 2, 3], [1, 2, 4], [1, 2, 5], [1, 2, 6], [1, 2, 7], [1, 2, 8], [1, 2, 9], [1, 2, 10], [1, 2, 11], [1, 2, 12]  …  [16, 17, 18], [16, 17, 19], [16, 17, 20], [16, 18, 19], [16, 18, 20], [16, 19, 20], [17, 18, 19], [17, 18, 20], [17, 19, 20], [18, 19, 20]]
 [[1, 2, 3, 4], [1, 2, 3, 5], [1, 2, 3, 6], [1, 2, 3, 7], [1, 2, 3, 8], [1, 2, 3, 9], [1, 2, 3, 10], [1, 2, 3, 11], [1, 2, 3, 12], [1, 2, 3, 13]  …  [15, 16, 19, 20], [15, 17, 18, 19], [15, 17, 18, 20], [15, 17, 19, 20], [15, 18, 19, 20], [16, 17, 18, 19], [16, 17, 18, 20], [16, 17, 19, 20], [16, 18, 19, 20], [17, 18, 19, 20]]

In [ ]:
# Use all_i_neq_j_index_combinations, all_i_neq_j_str_combinations and indexed_DE_with_sampled_combinations
# to construct DE terms 

In [87]:
op_index_vec

62-element Vector{String}:
 "-"
 "x"
 "y"
 "z"
 "--"
 "+-"
 "-x"
 "-y"
 "-z"
 "xx"
 ⋮
 "zxx"
 "zxy"
 "zxz"
 "zyx"
 "zyy"
 "zyz"
 "zzx"
 "zzy"
 "zzz"

(["-", "x", "y", "z", "--", "+-", "-x", "-y", "-z", "---", "+--", "--x", "+-x", "--y", "+-y", "--z", "+-z"], [["g"], ["\\beta", "\\sqrt{\\kappa}"], ["\\kappa"], ["\\Delta"], ["\\gamma"], ["\\Gamma"], ["\\beta^{*}", "\\sqrt{\\kappa}"]], ["---z", "+--z", "---y", "---x", "+--y", "+--x"])

In [65]:
#### Calculate Cumulant terms from cumulant_indexed_vector and an operator expectation value vector
# Test
p::Float64 = 0.1
alpha::Float64 = 0.1
Theta::Float64 = 0.1
all_eqs = single_spin_gen_DE_up_to_order(3)
conjugate_to_basis!(all_eqs)
all_eqs_indexed, op_index_dicts, vars_dict, cumulant_terms_dict = parse_DE_to_indexes(all_eqs)
cumulant_vector = cumulant_terms_from_dict(cumulant_terms_dict, op_index_dicts)
lower_cumulants = lower_order_full_cumulant_terms(op_index_dicts, 2)
op_index_vec, vars_vec, cumulant_terms_vec = preprocess_index_inversion(op_index_dicts, vars_dict, cumulant_terms_dict)
initial_conditions = equations2initial_conditions(op_index_vec, p, alpha, Theta)
display(cumulant_vector)
display(lower_cumulants)
display(get_cumulant_values_provided(cumulant_vector, initial_conditions))

6-element Vector{Cumulant_indexed}:
 Cumulant_indexed([[1, 1, 1, 4], [5, 1, 4], [9, 1, 1], [5, 9], [9, 5], [10, 4], [16, 1]], [6, -6, -6, 2, 1, 1, 3], "---z")
 Cumulant_indexed([[-1, 1, 1, 4], [6, 1, 4], [-9, 1, 1], [5, -1, 4], [9, -1, 1], [6, 9], [-9, 5], [11, 4], [17, 1], [16, -1]], [6, -4, -2, -2, -4, 2, 1, 1, 2, 1], "+--z")
 Cumulant_indexed([[1, 1, 1, 3], [5, 1, 3], [8, 1, 1], [5, 8], [8, 5], [10, 3], [14, 1]], [6, -6, -6, 2, 1, 1, 3], "---y")
 Cumulant_indexed([[1, 1, 1, 2], [5, 1, 2], [7, 1, 1], [5, 7], [7, 5], [10, 2], [12, 1]], [6, -6, -6, 2, 1, 1, 3], "---x")
 Cumulant_indexed([[-1, 1, 1, 3], [6, 1, 3], [-8, 1, 1], [5, -1, 3], [8, -1, 1], [6, 8], [-8, 5], [11, 3], [15, 1], [14, -1]], [6, -4, -2, -2, -4, 2, 1, 1, 2, 1], "+--y")
 Cumulant_indexed([[-1, 1, 1, 2], [6, 1, 2], [-7, 1, 1], [5, -1, 2], [7, -1, 1], [6, 7], [-7, 5], [11, 2], [13, 1], [12, -1]], [6, -4, -2, -2, -4, 2, 1, 1, 2, 1], "+--x")

3-element Vector{Vector{Vector{Full_Cumulant_indexed}}}:
 [[Full_Cumulant_indexed([[1]], [1], "-")], [Full_Cumulant_indexed([[4]], [1], "z"), Full_Cumulant_indexed([[3]], [1], "y")]]
 [[Full_Cumulant_indexed([[-1, 1], [6]], [-1, 1], "+-"), Full_Cumulant_indexed([[1, 1], [5]], [-1, 1], "--")], [Full_Cumulant_indexed([[1, 4], [9]], [-1, 1], "-z"), Full_Cumulant_indexed([[1, 3], [8]], [-1, 1], "-y")]]
 [[Full_Cumulant_indexed([[-1, 1, 1], [6, 1], [5, -1], [11]], [2, -2, -1, 1], "+--"), Full_Cumulant_indexed([[1, 1, 1], [5, 1], [10]], [2, -3, 1], "---")], [Full_Cumulant_indexed([[-1, 1, 4], [6, 4], [-9, 1], [9, -1], [17]], [2, -1, -1, -1, 1], "+-z"), Full_Cumulant_indexed([[-1, 1, 3], [6, 3], [-8, 1], [8, -1], [15]], [2, -1, -1, -1, 1], "+-y")]]

6-element Vector{ComplexF64}:
 0.0007642691913004843 - 0.00023641616532907176im
 0.0007960033322224217 - 7.986673331746262e-5im
                   0.0 + 0.0im
                   0.0 + 0.0im
                   0.0 + 0.0im
                   0.0 + 0.0im

In [37]:
# Construct distributions of parameters rho(g) 
# Sample from said distributions
# Construct basis functions b_i(g) on space (of sampled parameters)
# integrate int_V N*rho(g) f(g) b_i(g) dg = B_i    (here N is the total number of spins)
# Use linear regression to construct 
